# Pneumonia-X — étape 4 : validation externe sur RSNA (adultes)

Avant de lancer :
1. Accepter les règles de la compétition **RSNA Pneumonia Detection Challenge** (bouton *Join Competition*).
2. *Session options* : **GPU T4 x2**, **Internet On**.
3. **Add Input** : `Chest X-Ray Images (Pneumonia)` (paultimothymooney) **et** la compétition `RSNA Pneumonia Detection Challenge`.
4. Lancer avec **Save Version → Save & Run All (Commit)** : le calcul tourne en arrière-plan et les fichiers de sortie
   (dont les checkpoints, réutilisés aux étapes 5 et 6) sont conservés. Durée : environ 45 à 60 min.

In [ ]:
!nvidia-smi -L
import glob, os
for p in glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*'):
    print(p)

In [ ]:
%cd /kaggle/working
!rm -rf pneumonia && git clone -q https://github.com/khadijafair/pneumonia.git
%cd /kaggle/working/pneumonia
!git log --oneline -1
%env PYTHONPATH=/kaggle/working/pneumonia/src
%env MLFLOW_DISABLE_AGENT_HINT=1
!pip install -q -r requirements-train.txt

## 1. Réentraîner les 3 modèles (mêmes réglages qu'à l'étape 2)

In [ ]:
DATA = "/kaggle/input"
!python -m pneumonia.train -m model=efficientnet_b0,densenet121,vit_small data.root={DATA}

## 2. Évaluation interne de cette exécution (enfants, test par patient)

In [ ]:
import shutil, pathlib
res = pathlib.Path('results_rsna'); shutil.rmtree(res, ignore_errors=True); res.mkdir()
ckpts = []
for d in sorted(pathlib.Path('outputs').iterdir()):
    if (d / 'best.ckpt').exists():
        (res / d.name).mkdir()
        for f in [d / 'summary.json', *d.glob('pred_*.csv')]:
            shutil.copy(f, res / d.name)
        ckpts.append(str(d / 'best.ckpt'))
print(ckpts)
!python scripts/evaluate.py --results results_rsna

## 3. Prédictions sur RSNA (environ 26 700 radios d'adultes, DICOM)

In [ ]:
!python scripts/predict_external.py --rsna {DATA} --ckpt {' '.join(ckpts)} --results results_rsna

## 4. Évaluation externe

In [ ]:
!python scripts/evaluate_external.py --results results_rsna

## 5. Sauvegarde : checkpoints (pour les étapes 5–6) et résultats à télécharger

In [ ]:
ck = pathlib.Path('/kaggle/working/checkpoints'); ck.mkdir(exist_ok=True)
for c in ckpts:
    shutil.copy(c, ck / f"{pathlib.Path(c).parent.name}.ckpt")
!cd /kaggle/working/pneumonia && zip -qr /kaggle/working/results_step4.zip results_rsna mlflow.db mlartifacts
!ls -la /kaggle/working/checkpoints /kaggle/working/results_step4.zip